# 07 · 200 ms EMG windows with a 50 ms step
Does a longer window improve the best approach so far (patch-embedding RoFormer, absolute + delta heads, Kalman fusion)?

Data are preprocessed once with **400-sample (200 ms) windows and a 100-sample (50 ms) step**, same causal filter,
repetition split and train-only normalisation as before. Three models are trained on them:

| name | window seen by the model | tokens | purpose |
|---|---|---|---|
| `w100ms_patch8_t25` | last 100 ms of each window | 25 | **control**: today's model, same 50 ms step and the exact same test windows |
| `w200ms_patch16_t25` | 200 ms | 25 | longer window, about the same hardware cost as today |
| `w200ms_patch8_t50` | 200 ms | 50 | longer window, finer tokens, about 2x the compute |

The control isolates the effect of window length: every model is scored on windows that end at the same samples.
The Kalman filter is re-tuned per model because the step is now 50 ms instead of 25 ms.

**Before running:** re-upload `emg_common.py` (adds window cropping). Attach NinaPro DB2. GPU on. Expected time about 2.5 to 3 hours.
Runs resume: attach this notebook's saved output and re-run; finished preprocessing and checkpoints are reused. Click **Save Version** at the end.

In [ ]:
import os, sys, glob, json, time
for p in ['/kaggle/usr/lib/notebooks/quddusikashaf/emg_common', '/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '.', '..', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
assert hasattr(ec, 'crop_last') and hasattr(ec, 'per_subject_eval'), 're-upload the updated emg_common.py utility script'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')
N_BOOT = int(os.environ.get('EMG_NBOOT', 2000))
EPOCHS = int(os.environ.get('EMG_EPOCHS', 40))
QAT_EPOCHS = int(os.environ.get('EMG_QAT', 0))         # 0: float comparison; set 5 for the model you will deploy
OUT = f'{WORK}/w200'; os.makedirs(f'{OUT}/outputs', exist_ok=True)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| emg_common', ec.__file__)

## 1 · Preprocess with 200 ms windows and a 50 ms step

In [ ]:
WIN, STEP = 400, 100                       # samples at 2 kHz
TAG = f'db2_processed_w{WIN}_s{STEP}'
BASE = os.environ.get('EMG_BASE_PATH')
if BASE is None:
    hits = sorted(glob.glob('/kaggle/input/**/DB2_s1/DB2_s1', recursive=True))
    BASE = os.path.dirname(os.path.dirname(hits[0])) if hits else ec.Config.base_path
dcfg = ec.Config(base_path=BASE, processed_dir=f'{WORK}/{TAG}', seq_len=WIN, stride=STEP,
                 n_subjects=int(os.environ.get('EMG_N_SUBJECTS', 10)), causal_filter=True, train_reps=(1, 3, 4, 6), test_reps=(2, 5))
subjects = ec.list_subjects(dcfg)
reuse = sorted(glob.glob(f'/kaggle/input/**/{TAG}', recursive=True))
npz_files = []
for s in subjects:
    found = [f'{d}/{s}/data.npz' for d in [dcfg.processed_dir] + reuse if os.path.exists(f'{d}/{s}/data.npz')]
    npz_files.append(found[0] if found else ec.process_subject(dcfg, s))
d0 = np.load(npz_files[0])
print(len(npz_files), 'subjects | window', d0['sEMG'].shape[1], 'samples | windows per subject ~', len(d0['split']),
      '| train', int((d0['split'] == 0).sum()), 'test', int((d0['split'] == 1).sum()))
print('per-joint std of the 50 ms delta (train):', np.round(ec.delta_scale_from_train(npz_files), 4))

## 2 · Train (resumable)

In [ ]:
EXPERIMENTS = [
    dict(name='w100ms_patch8_t25',  seq_len=200, patch_stride=8),    # control
    dict(name='w200ms_patch16_t25', seq_len=400, patch_stride=16),
    dict(name='w200ms_patch8_t50',  seq_len=400, patch_stride=8),
]
if 'EMG_EXPERIMENTS' in os.environ:
    keep = os.environ['EMG_EXPERIMENTS'].split(','); EXPERIMENTS = [e for e in EXPERIMENTS if e['name'] in keep]

def make_cfg(e):
    c = ec.Config(processed_dir=os.path.dirname(os.path.dirname(npz_files[0])), save_dir=f'{OUT}/outputs',
                  variant='patch', patch_stride=e['patch_stride'], seq_len=e['seq_len'], stride=STEP,
                  heads='both', cond_prev=False, unroll_steps=0, d_model=128, d_ff=256, num_heads=8, num_layers=1,
                  norm='layernorm', dropout=0.1, epochs=EPOCHS, patience=8, lr=3e-4, batch_size=64,
                  qat_epochs=QAT_EPOCHS, act_fmt=(10, 3), resid_fmt=(16, 8), weight_fmt=(10, 3),
                  loss_norm=True, select_metric='val_abs', use_wandb=os.environ.get('EMG_WANDB', '1') == '1',
                  project_name='ninapro_db2_window200')
    c.apply_variant(); c.run_name = e['name']
    return c

device = ec.get_device(); models = {}
for e in EXPERIMENTS:
    c = make_cfg(e)
    print(f"\n=== {e['name']}: {c.seq_len} samples -> {c.num_tokens} tokens | MACs {ec.estimate_macs(c)['total']:,}")
    found = [p for p in [f'{OUT}/outputs/{e["name"]}.pt'] + sorted(glob.glob(f'/kaggle/input/**/{e["name"]}.pt', recursive=True)) if os.path.exists(p)]
    if found:
        m, c, extra = ec.load_checkpoint(found[0], device); print('reusing', found[0], extra.get('phase'))
    else:
        run = ec.init_wandb(c)
        t0 = time.time(); m, info = ec.train_model(c, npz_files, run=run, tag=e['name'])
        print(f'trained in {(time.time() - t0) / 60:.1f} min')
        if run is not None: run.finish()
    models[e['name']] = (m.eval(), c)

## 3 · Evaluate: re-tuned filters, per-subject bootstrap, window metrics

In [ ]:
rows, per_subject = [], []
for name, (m, c) in models.items():
    preds = [ec.predict_run(m, r, c, device, feedback=True) for f in npz_files for r in ec.ordered_runs(f, 'test') if len(r['angles']) >= 4]
    gk = ec.kalman_sweep(preds, use_delta=True, init='abs'); gs = ec.kalman_sweep(preds, use_delta=False, init='abs')
    parse = lambda key: (float(key[0].split('=')[1]), float(key[1].split('=')[1]))
    kqr, sqr = parse(gk.stack().idxmin()), parse(gs.stack().idxmin())
    df = ec.per_subject_eval(m, npz_files, c, device, init='abs', kalman_qr=kqr, smooth_qr=sqr, n_boot=N_BOOT, tag=name)
    per_subject.append(df)
    _, val_loader = ec.make_loaders(c, npz_files)
    wm = ec.window_metrics(ec.predict_windows(m, val_loader, c, device))
    a = df[df['subject'] == 'ALL'].iloc[0]; macs = ec.estimate_macs(c)
    rows.append({'model': name, 'window_ms': c.seq_len / 2, 'step_ms': STEP / 2, 'tokens': c.num_tokens,
                 'params': ec.count_params(m), 'MACs_M': macs['total'] / 1e6, 'dense_bound_ms_U4': macs['total'] / 4 / 1e5,
                 'win_abs_r2': wm.get('abs_r2'), 'win_delta_r2': wm.get('delta_r2'),
                 'abs_only_r2': a['abs_only_r2'], 'smoothed_r2': a['abs_smoothed_r2'],
                 'fused_r2': a['kalman_r2'], 'fused_r2_lo': a['kalman_r2_lo'], 'fused_r2_hi': a['kalman_r2_hi'],
                 'fused_rmse': a['kalman_rmse'], 'delta_gain': a['delta_gain'], 'delta_gain_lo': a['delta_gain_lo'],
                 'delta_gain_hi': a['delta_gain_hi'], 'kalman_qr': kqr, 'smooth_qr': sqr})
    print(f"{name}: fused R2 {a['kalman_r2']:.4f} [{a['kalman_r2_lo']:.4f}, {a['kalman_r2_hi']:.4f}] | abs-only {a['abs_only_r2']:.4f} | best (q, r) {kqr}")
summary = pd.DataFrame(rows); per_subject = pd.concat(per_subject, ignore_index=True)
summary.to_csv(f'{OUT}/summary.csv', index=False); per_subject.to_csv(f'{OUT}/per_subject.csv', index=False)
display(summary.round(4))
print('Reference, session 3 (100 ms window, 25 ms step, different test windows): fused R2 0.667, abs-only 0.555, 3.75 M MACs.')

In [ ]:
# ---- paired comparison against the control, per subject ----
piv = per_subject[per_subject['subject'] != 'ALL'].pivot(index='subject', columns='setting', values='kalman_r2')
ctrl = 'w100ms_patch8_t25'
if ctrl in piv:
    for col in piv.columns:
        if col == ctrl: continue
        d = piv[col] - piv[ctrl]
        print(f'{col} vs control: mean R2 change {d.mean():+.4f}, better in {(d > 0).sum()} of {len(d)} subjects')
display(piv.round(4))
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(summary))
ax[0].bar(x, summary['fused_r2'], yerr=[summary['fused_r2'] - summary['fused_r2_lo'], summary['fused_r2_hi'] - summary['fused_r2']], capsize=4, label='fused')
ax[0].scatter(x, summary['abs_only_r2'], color='k', marker='x', zorder=3, label='abs head only')
ax[0].set_xticks(x); ax[0].set_xticklabels(summary['model'], rotation=15); ax[0].set_ylabel('R² (pooled, 95 % CI)'); ax[0].legend(); ax[0].grid(alpha=.3, axis='y')
piv.plot(ax=ax[1], marker='o'); ax[1].set_ylabel('fused R² per subject'); ax[1].grid(alpha=.3); ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig(f'{OUT}/window_comparison.png', dpi=120); plt.show()
json.dump({'win': WIN, 'step': STEP, 'n_boot': N_BOOT, 'qat_epochs': QAT_EPOCHS, 'summary': summary.to_dict(orient='records')},
          open(f'{OUT}/summary.json', 'w'), indent=2, default=str)

### Reading the result
* Compare each 200 ms model with **the control row**, not with session 3: only the control shares the step and the test windows.
* A gain is credible if the pooled confidence intervals separate and most subjects improve in the paired table.
* Hardware: `w200ms_patch16_t25` keeps 25 tokens, so the HLS design is unchanged except `RAW_SEQ_LEN=400`, `KERNEL_SIZE=CONV_STRIDE=16`
  (input 4,800 words, embedding 2x the MACs). `w200ms_patch8_t50` doubles the token count: about 2x the dense-layer time and 4x the attention time.
  Latency budget is now 50 ms per step.